# Phase 2: Feature Engineering & Data Preprocessing

## 2.1 Data Segregation (Train-Test Split)

### Objective
To isolate the independent variables ($X$) from the dependent target variable ($y$), and partition the dataset into training and testing environments prior to any transformations to prevent data leakage.

### Rationale
We utilize a standard 80/20 split. Because our EDA revealed a class imbalance in the target variable (~20% churn rate), we apply `stratify=y` to ensure this exact distribution is mathematically preserved across both the training and testing sets. We also drop `customer_id` as it holds no predictive mathematical value.

In [1]:
# 1. Import required libraries
import pandas as pd
from sklearn.model_selection import train_test_split

# 2. Load the raw data (Using the safe relative path)
df = pd.read_csv('../data/bank_churn.csv')

# 3. Drop the non-mathematical identifier
df = df.drop(['customer_id'], axis=1)

# 4. Separate Features (X) from Target (y)
X = df.drop(['churn'], axis=1)
y = df['churn']

# 5. Execute the Train-Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 6. Verify the split
print(f"Original Dataset Shape: {df.shape}")
print(f"Training Features (X_train) Shape: {X_train.shape}")
print(f"Testing Features (X_test) Shape: {X_test.shape}")

Original Dataset Shape: (10000, 11)
Training Features (X_train) Shape: (8000, 10)
Testing Features (X_test) Shape: (2000, 10)


## 2.2 Categorical Encoding

### Objective
Translate text-based categorical features (`gender`, `country`) into numerical formats without introducing unintended mathematical hierarchy.

### Strategy
* **Binary Mapping:** `gender` is mapped directly to 1 (Male) and 0 (Female).
* **One-Hot Encoding:** `country` is encoded into distinct binary columns. We utilize `drop='first'` to avoid the Dummy Variable Trap (perfect multicollinearity).

In [2]:
from sklearn.preprocessing import OneHotEncoder

# --- 1. BINARY ENCODING (Gender) ---
# Direct dictionary mapping is fastest for binary choices
gender_mapping = {'Male': 1, 'Female': 0}

X_train['gender'] = X_train['gender'].map(gender_mapping)
X_test['gender'] = X_test['gender'].map(gender_mapping)

# --- 2. ONE-HOT ENCODING (Country) ---
# drop='first' prevents multicollinearity (if not Spain and not Germany, it MUST be France)
encoder = OneHotEncoder(sparse_output=False, drop='first')

# FIT the encoder ONLY on the training data to prevent data leakage
encoder.fit(X_train[['country']])

# Get the new column names (e.g., 'country_Germany', 'country_Spain')
ohe_columns = encoder.get_feature_names_out(['country'])

# TRANSFORM both train and test sets
train_encoded_array = encoder.transform(X_train[['country']])
test_encoded_array = encoder.transform(X_test[['country']])

# Convert the raw arrays back into Pandas DataFrames, ensuring the index matches
X_train_encoded = pd.DataFrame(train_encoded_array, columns=ohe_columns, index=X_train.index)
X_test_encoded = pd.DataFrame(test_encoded_array, columns=ohe_columns, index=X_test.index)

# Drop the old 'country' text column and glue the new numerical columns onto the dataset
X_train = pd.concat([X_train.drop('country', axis=1), X_train_encoded], axis=1)
X_test = pd.concat([X_test.drop('country', axis=1), X_test_encoded], axis=1)

# Verify the transformation
print("Successfully Encoded X_train. First 3 rows:")
display(X_train.head(3))

Successfully Encoded X_train. First 3 rows:


,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,country_Germany,country_Spain
2151,753,1,57,7,0.00,1,1,0,159475.08,0.0,0.0
8392,739,1,32,3,102128.27,1,1,0,63981.37,1.0,0.0
5006,755,0,37,0,113865.23,2,1,1,117396.25,1.0,0.0


## 2.3 Feature Scaling

### Objective
Standardize continuous numerical features to ensure algorithms evaluate them equally, preventing features with massive raw numbers (e.g., `balance`) from dominating the distance calculations.

### Strategy
We utilize `StandardScaler` to transform continuous features so they have a mean of 0 and a standard deviation of 1. We strictly exclude our binary/encoded categorical features (`gender`, `country_...`) from this scaling process to preserve their distinct 1/0 boolean logic. The scaler is fitted exclusively on the training set to prevent data leakage.

In [3]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Scaler
scaler = StandardScaler()

# 2. Define exactly which columns need scaling (Continuous numerical variables only)
# We deliberately leave out 'gender' and our newly one-hot encoded country columns
cols_to_scale = ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary']

# 3. Fit on the Training set, and Transform both sets
# We overwrite the original columns with the newly scaled data
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# 4. Verify the transformation
print("Successfully Scaled Continuous Features. First 3 rows:")
display(X_train.head(3))

Successfully Scaled Continuous Features. First 3 rows:


,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,country_Germany,country_Spain
2151,1.058568,1,1.715086,0.684723,-1.226059,-0.910256,1,0,1.042084,0.0,0.0
8392,0.913626,1,-0.659935,-0.696202,0.413288,-0.910256,1,0,-0.623556,1.0,0.0
5006,1.079274,0,-0.184931,-1.731895,0.601687,0.808830,1,1,0.308128,1.0,0.0


## 2.4 Target Balancing (SMOTE)

### Objective
Address the 80/20 class imbalance in the target variable (`churn`) to prevent the model from developing a majority-class bias.

### Strategy
We apply SMOTE (Synthetic Minority Over-sampling Technique) to generate statistically realistic synthetic examples of the minority class (Churn = 1). 
**Crucial MLOps Constraint:** SMOTE is applied *exclusively* to the Training Set. The Testing Set must retain its natural real-world distribution to provide a valid evaluation metric.

In [5]:
from imblearn.over_sampling import SMOTE

# 1. Initialize SMOTE
# random_state ensures reproducibility
smote = SMOTE(random_state=42)

# 2. Fit and Transform ONLY the Training Set
# This generates new synthetic rows for X_train and corresponding '1's for y_train
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# 3. Verify the transformation
print("Original y_train Class Distribution:")
print(y_train.value_counts(normalize=True))

print("\nNew y_train_balanced Class Distribution:")
print(y_train_balanced.value_counts(normalize=True))

print(f"\nNew Training Set Shape: {X_train_balanced.shape}")

Original y_train Class Distribution:
churn
0    0.79625
1    0.20375
Name: proportion, dtype: float64

New y_train_balanced Class Distribution:
churn
1    0.5
0    0.5
Name: proportion, dtype: float64

New Training Set Shape: (12740, 11)


## 2.5 Feature Engineering Conclusion & Pipeline State

### Summary of Transformations
We have successfully transformed the raw, imbalanced dataset into a mathematically optimized matrix, strictly adhering to MLOps leakage-prevention protocols. The following pipeline was executed:

1. **Data Segregation:** Isolated 20% of the data as a pure testing environment (`X_test`), stratified to preserve the natural 80/20 churn distribution.
2. **Categorical Encoding:** Translated string variables into mathematical arrays using Binary Mapping (`gender`) and One-Hot Encoding (`country`), explicitly dropping the first column to prevent multicollinearity (the Dummy Variable Trap).
3. **Feature Scaling:** Applied `StandardScaler` to continuous features (e.g., `balance`, `age`) to equalize their mathematical weight for distance-based algorithms, deliberately bypassing our boolean logic features.
4. **Target Balancing:** Deployed SMOTE strictly on the training set (`X_train`) to synthetically resolve the 80/20 class imbalance. The training matrix is now a perfectly balanced 50/50 distribution (12,740 rows), ensuring the model learns churn patterns without majority-class bias.

### Pipeline Readiness
The feature engineering phase is formally concluded. The data matrices (`X_train`, `X_test`, `y_train_balanced`, `y_test`) are mathematically stable and ready for Phase 3: Model Training & Evaluation.